# 02 · Población vs. restaurantes (Censo 2020 + DENUE 2020)

**Objetivo:** construir la tabla base por hexágono para 2020 y ver dónde vive la gente contra dónde están los restaurantes.

- **Censo 2020 por manzana:** población, viviendas, empleo, autos, escolaridad.
- **DENUE 04/2020:** restaurantes por tipo y total de establecimientos (contexto).
- **Unidad común:** cada manzana se asigna al hexágono H3 que contiene su centroide.

La tabla que sale de aquí (`hex_res8_base_2020.parquet`) es el punto de partida de las variables del modelo.

## 0. Librerías y rutas

In [ ]:
from pathlib import Path
import tempfile, time, zipfile, urllib.request
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import h3
import mapas

ROOT = Path.cwd().parent
RAW, PROC, FIG = ROOT / "data/raw", ROOT / "data/processed", ROOT / "figures"
for p in (RAW, PROC, FIG):
    p.mkdir(parents=True, exist_ok=True)
TMP = Path(tempfile.gettempdir()) / "inegi_tmp"
TMP.mkdir(exist_ok=True)

URL_CENSO = "https://www.inegi.org.mx/contenidos/programas/ccpv/2020/datosabiertos/ageb_manzana/ageb_mza_urbana_22_cpv2020_csv.zip"
URL_MG = "https://www.inegi.org.mx/contenidos/productos/prod_serv/contenidos/espanol/bvinegi/productos/geografia/marcogeo/889463807469/22_queretaro.zip"

MUNICIPIOS_ZMQ = {6: "Corregidora", 8: "Huimilpan", 11: "El Marqués", 14: "Querétaro"}
RES = 8
EDICION_BASE = "2020-04"

## 1. Descarga

El CSV del Censo (unos MB) se guarda en `data/raw/`. El Marco Geoestadístico (62.7 MB) se baja a una carpeta temporal fuera de Dropbox.

In [ ]:
def descargar(url, destino, intentos=3):
    if destino.exists():
        print(f"Ya existe: {destino.name}")
        return destino
    for i in range(intentos):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=300) as r, open(destino, "wb") as f:
                while bloque := r.read(1 << 20):
                    f.write(bloque)
            print(f"Descargado: {destino.name} ({destino.stat().st_size / 1e6:.1f} MB)")
            return destino
        except Exception as e:
            print(f"  intento {i + 1} falló: {e}")
            time.sleep(5)
    raise RuntimeError(f"No se pudo descargar {url}")

zip_censo = descargar(URL_CENSO, RAW / URL_CENSO.split("/")[-1])

## 2. Manzanas: centroides de la zona metropolitana

Del Marco Geoestadístico se extrae solo la capa de manzanas (`22m`). El centroide se calcula en la proyección original (metros) y luego se pasa a latitud y longitud. Se guarda en `data/processed/manzanas_zmq_2020.parquet`; si ya existe, no se vuelve a descargar.

In [ ]:
SALIDA_MZA = PROC / "manzanas_zmq_2020.parquet"

if SALIDA_MZA.exists():
    mza = pd.read_parquet(SALIDA_MZA)
    print("Centroides leídos de", SALIDA_MZA.name)
else:
    zip_mg = descargar(URL_MG, TMP / "22_queretaro.zip")
    destino = TMP / "mg_22"
    with zipfile.ZipFile(zip_mg) as z:
        capa = [n for n in z.namelist() if Path(n).stem.lower() == "22m"]
        if not capa:
            raise FileNotFoundError("No encontré la capa 22m (manzanas) en el zip del Marco Geoestadístico")
        z.extractall(destino, members=capa)
    gdf = gpd.read_file(next(destino.rglob("22m.shp")))
    gdf.columns = [c.upper() if c != "geometry" else c for c in gdf.columns]
    gdf = gdf[pd.to_numeric(gdf["CVE_MUN"]).isin(MUNICIPIOS_ZMQ)]
    cent_ll = gpd.GeoSeries(gdf.geometry.centroid, crs=gdf.crs).to_crs(4326)
    mza = pd.DataFrame({"CVEGEO": gdf["CVEGEO"].values, "lat": cent_ll.y.values,
                        "lon": cent_ll.x.values, "area_m2": gdf.geometry.area.values})
    mza.to_parquet(SALIDA_MZA, index=False)
    print(f"Manzanas de la ZM: {len(mza):,} → {SALIDA_MZA.name}")
mza.head()

## 3. Censo 2020 por manzana

Se quitan los renglones de totales (`MZA = 0`) y se arma `CVEGEO` = entidad (2) + municipio (3) + localidad (4) + AGEB (4) + manzana (3).

Variables que se conservan (todas son conteos, así que se pueden sumar por hexágono):

| Variable | Qué es |
|---|---|
| `POBTOT` | Población total |
| `TVIVHAB` | Viviendas habitadas |
| `POCUPADA` | Población ocupada |
| `VPH_AUTOM` | Viviendas con automóvil |
| `P_15YMAS`, `GRAPROES` | Población de 15+ y su escolaridad promedio (para un promedio ponderado) |

Los valores ocultos con `*` por confidencialidad quedan como faltantes, y se reporta cuántos son.

In [ ]:
with zipfile.ZipFile(zip_censo) as z:
    nombre = max((i for i in z.infolist() if i.filename.lower().endswith(".csv")), key=lambda i: i.file_size)
    censo = pd.read_csv(z.open(nombre), dtype=str, encoding="utf-8", low_memory=False)
censo.columns = censo.columns.str.upper()

censo = censo[pd.to_numeric(censo["MUN"]).isin(MUNICIPIOS_ZMQ) & (pd.to_numeric(censo["MZA"]) != 0)].copy()
censo["CVEGEO"] = (censo["ENTIDAD"].str.zfill(2) + censo["MUN"].str.zfill(3) + censo["LOC"].str.zfill(4)
                   + censo["AGEB"].str.zfill(4) + censo["MZA"].str.zfill(3))

VARS = [v for v in ["POBTOT", "TVIVHAB", "POCUPADA", "VPH_AUTOM", "P_15YMAS", "GRAPROES"] if v in censo.columns]
for v in VARS:
    censo[v] = pd.to_numeric(censo[v], errors="coerce")
if {"GRAPROES", "P_15YMAS"} <= set(VARS):
    censo["ESC_X_POB"] = censo["GRAPROES"] * censo["P_15YMAS"]      # para el promedio ponderado
print(f"Manzanas en el Censo (ZM): {len(censo):,} · población: {censo['POBTOT'].sum():,.0f}")
print("Valores ocultos o faltantes:", censo[VARS].isna().sum().to_dict())

## 4. Llevar todo a hexágonos

Revisa la tasa de cruce entre Censo y manzanas: idealmente cerca de 100%.

In [ ]:
mzc = censo.merge(mza, on="CVEGEO", how="inner")
print(f"Manzanas del Censo con polígono: {len(mzc) / len(censo):.1%}")
mzc["h3"] = [h3.latlng_to_cell(la, lo, RES) for la, lo in zip(mzc["lat"], mzc["lon"])]

SUMAR = [v for v in ["POBTOT", "TVIVHAB", "POCUPADA", "VPH_AUTOM", "P_15YMAS", "ESC_X_POB"] if v in mzc.columns]
pob_hex = mzc.groupby("h3")[SUMAR].sum(min_count=1)
if "ESC_X_POB" in pob_hex:
    pob_hex["escolaridad"] = pob_hex["ESC_X_POB"] / pob_hex["P_15YMAS"]
    pob_hex = pob_hex.drop(columns="ESC_X_POB")

def a_hex(df):
    return df.assign(h3=[h3.latlng_to_cell(la, lo, RES) for la, lo in zip(df["latitud"], df["longitud"])])

# Restaurantes 2020 por tipo (columnas anchas) y total de establecimientos (contexto)
rest = pd.read_parquet(PROC / "restaurantes_zmq.parquet")
rest = a_hex(rest[rest["edicion"] == EDICION_BASE])
rest_hex = rest.pivot_table(index="h3", columns="tipo", values="id", aggfunc="count", fill_value=0)
rest_hex.columns = ["rest_" + c for c in rest_hex.columns]
rest_hex["restaurantes"] = rest_hex.sum(axis=1)

den = a_hex(pd.read_parquet(PROC / f"denue_zmq_{EDICION_BASE}.parquet", columns=["latitud", "longitud"]))
est_hex = den.groupby("h3").size().rename("establecimientos")

hexes = pob_hex.join([est_hex, rest_hex], how="outer")
cols_conteo = [c for c in hexes.columns if c != "escolaridad"]
hexes[cols_conteo] = hexes[cols_conteo].fillna(0)
hexes = hexes[(hexes["POBTOT"] > 0) | (hexes["establecimientos"] > 0)]
print(f"Hexágonos con población o establecimientos: {len(hexes):,}")
hexes[["POBTOT", "establecimientos", "restaurantes"]].describe().round(1)

## 5. Clasificación: dónde vive la gente vs. dónde están los restaurantes

Cada hexágono se compara contra la mediana de población y la mediana de restaurantes (entre los hexágonos que tienen al menos uno):

| Población | Restaurantes | Tipo de zona |
|---|---|---|
| ≥ mediana | < mediana | Residencial con poca oferta |
| < mediana | ≥ mediana | Polo de restaurantes |
| ≥ mediana | ≥ mediana | Mixta |
| < mediana | < mediana | Baja densidad |

Las zonas *residenciales con poca oferta* son las más interesantes para la recomendación: hay gente, pero pocos restaurantes. Si es demanda sin atender o una zona poco favorable es justo lo que el modelo tiene que distinguir.

In [ ]:
p50 = hexes.loc[hexes["POBTOT"] > 0, "POBTOT"].median()
r50 = hexes.loc[hexes["restaurantes"] > 0, "restaurantes"].median()
alta_pob, alta_rest = hexes["POBTOT"] >= p50, hexes["restaurantes"] >= r50
TIPOS_ZONA = ["Residencial con poca oferta", "Polo de restaurantes", "Mixta", "Baja densidad"]
hexes["tipo_zona"] = np.select([alta_pob & ~alta_rest, ~alta_pob & alta_rest, alta_pob & alta_rest],
                               TIPOS_ZONA[:3], default=TIPOS_ZONA[3])

print(f"Medianas → población: {p50:,.0f} hab · restaurantes: {r50:,.0f}")
resumen = hexes.groupby("tipo_zona").agg(hexagonos=("tipo_zona", "size"), poblacion=("POBTOT", "sum"),
                                         restaurantes=("restaurantes", "sum")).reindex(TIPOS_ZONA)
resumen["% población"] = (resumen["poblacion"] / resumen["poblacion"].sum() * 100).round(1)
resumen["% restaurantes"] = (resumen["restaurantes"] / resumen["restaurantes"].sum() * 100).round(1)
resumen

## 6. Mapa A · Tipos de zona

In [ ]:
COLORES = {"Residencial con poca oferta": "#2E6F95", "Polo de restaurantes": "#E8822F",
           "Mixta": "#14213D", "Baja densidad": "#D9D3C7"}

fig, ax, polys = mapas.base_mapa(hexes.index)
mapas.capa_hex(ax, polys, facecolors=hexes["tipo_zona"].map(COLORES).values)
mapas.referencias(ax)
ax.legend(handles=[Patch(color=c, label=f"{t} ({resumen.loc[t, 'hexagonos']:,.0f} hex.)") for t, c in COLORES.items()],
          loc="lower left", frameon=True, fontsize=11, title="Tipo de zona")
ax.set_title(f"Dónde vive la gente vs. dónde están los restaurantes · 2020", loc="left", fontsize=13)
plt.tight_layout(); plt.savefig(FIG / "04_tipos_de_zona_restaurantes.png", dpi=200, bbox_inches="tight"); plt.show()

## 7. Mapa B · Restaurantes por cada mil habitantes

Solo hexágonos con al menos 100 habitantes; con menos, la razón se dispara y no dice nada.

In [ ]:
con_pob = hexes[hexes["POBTOT"] >= 100].copy()
con_pob["rest_x_mil"] = con_pob["restaurantes"] / con_pob["POBTOT"] * 1000

BORDES = [0, 1, 3, 6, 12, 25, max(26, con_pob["rest_x_mil"].max() + 1)]
ETIQ = ["0", "1–2", "3–5", "6–11", "12–24", "25 o más"]
cmap, norm = mapas.escala_por_clases(BORDES, ["#EAF1F6", "#BFD6E6", "#8DB7D3", "#E8A25F", "#C2410C", "#7F2704"])

fig, ax, polys = mapas.base_mapa(con_pob.index)
coll = mapas.capa_hex(ax, polys, array=con_pob["rest_x_mil"].values, cmap=cmap, norm=norm)
mapas.referencias(ax)
mapas.barra_clases(fig, ax, coll, BORDES, ETIQ, "Restaurantes por cada mil habitantes")
ax.set_title("Oferta de restaurantes por habitante · 2020", loc="left", fontsize=13)
plt.tight_layout(); plt.savefig(FIG / "05_restaurantes_por_mil_hab.png", dpi=200, bbox_inches="tight"); plt.show()
print(f"Mediana de la ZM: {con_pob['rest_x_mil'].median():.1f} restaurantes por mil habitantes")

## 8. Guardar la tabla base por hexágono

In [ ]:
salida = PROC / f"hex_res{RES}_base_2020.parquet"
hexes.reset_index().rename(columns={"index": "h3"}).to_parquet(salida, index=False)
print(f"Guardado: {salida.name} · {hexes.shape[0]:,} hexágonos × {hexes.shape[1]} columnas")